In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()

while repo_root.name != "SpaceCore":
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

In [2]:
import numpy as np
import spacecore as sc
import jax
import jax.numpy as jnp
from spacecore import HermitianSpace
from spacecore.kernels import dispatch_mode

jax.config.update("jax_enable_x64", True)

/home/ppelikh/miniconda3/envs/py311/lib/python3.11/site-packages/cupy/_environment.py:663: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


In [3]:
sc.set_context('cupy', float, check_level='standard')
sc.get_context()

Context(ops=CuPyOps(family='cupy'), dtype=dtype('float64'), check_level='standard')

In [4]:
import numpy as np
import spacecore as sc

ctx = sc.Context(sc.NumpyOps(), dtype=np.float64)
X = sc.DenseCoordinateSpace((2,), ctx)
A = sc.DiagonalLinOp(ctx.asarray([2.0, 3.0]), X, ctx)

print(A.apply(ctx.asarray([1.0, 2.0])))
print(A.rapply(ctx.asarray([1.0, 1.0])))

M = sc.DenseLinOp(ctx.asarray([[1.0, 2.0], [0.0, 1.0]]), X, X, ctx)
print((M.H @ M).is_hermitian())

[2. 6.]
[2. 3.]
True


In [22]:
d = 100
dom = sc.ElementwiseJordanSpace((d,))
cod = sc.ElementwiseJordanSpace((d,))

dom == cod

True

In [26]:
ctx = sc.get_context()
A = np.random.rand(2, d, d, d, d)
A = A[0, :, :, :, :] + 1j * A[1, :, :, :, :]
# A = (A + A.T.conj()) / 2
A = ctx.asarray(A.real)
# linop = sc.DenseLinOp(A, dom, cod, ctx=ctx)

# linop

In [109]:
d = 250
ctx = sc.get_context()
A = np.random.rand(10, d, d)
A = (A + A.swapaxes(-1, -2)) / 2
A = ctx.asarray(A)

In [110]:
import cupy as cp, cupyx
import cupyx.cusolver

In [111]:
%timeit cp.linalg.eigh(A)

20.4 ms ± 42.9 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [117]:
f1 = jax.jit(lambda M: jax.lax.linalg.eigh(M, implementation='QR', symmetrize_input=True, sort_eigenvalues=False, lower=False)[0])
f2 = jax.jit(lambda M: jax.lax.linalg.eigh(M, implementation='JACOBI', symmetrize_input=True, sort_eigenvalues=False, lower=False)[0])
f3 = jax.jit(lambda M: jax.experimental.sparse.linalg.lobpcg_standard(M, M[:, :1])[0])
f4 = jax.jit(lambda M: jnp.linalg.eigvalsh(M))

A = jnp.asarray(A)

In [118]:
%timeit (A @ A).block_until_ready()

85.3 μs ± 108 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [119]:
%timeit f1(A).block_until_ready()

7.36 ms ± 9.29 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [120]:
%timeit f2(A).block_until_ready()

7.35 ms ± 1.06 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [121]:
%timeit f4(A).block_until_ready()

7.34 ms ± 1.44 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
from spacecore.linalg import cg, lanczos_smallest, expm_multiply

In [10]:
b = dom.zeros() + 1
x0 = dom.zeros() + 2

In [13]:
f = jax.jit(cg)
with dispatch_mode('off'):
    f(linop, b)
    %timeit f(linop, b)

202 μs ± 158 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
f = jax.jit(cg)
with dispatch_mode('on'):
    f(linop, b)
    %timeit f(linop, b)

202 μs ± 192 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%timeit cg(linop, b)

136 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
%timeit lanczos_smallest(linop, b, max_iter=2)

470 ms ± 3.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
l_s = jax.jit(lambda L, _b: lanczos_smallest(linop, b, max_iter=2))
l_s(linop, b)

with dispatch_mode('on'):
    %timeit l_s(linop, b)

501 μs ± 681 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
f = jax.jit(expm_multiply)
%timeit f(linop, b, 1.)

12.3 ms ± 4.82 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
linop.H

AdjointLinOp(DenseLinOp(ℝ^(40, 40) → ℝ^(40, 40)).H, backend='jax', dtype=float64)

In [24]:
# @jax.jit
def f2(x):
    return A @ x

In [25]:
zeros = dom.zeros()
%timeit f2(zeros + 2.)

105 μs ± 29.4 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [24]:
dom.spectral_apply(zeros + .5, dom.ops.sqrt)

Array([0.70710678+0.j, 0.70710678+0.j, 0.70710678+0.j, 0.70710678+0.j,
       0.70710678+0.j, 0.70710678+0.j, 0.70710678+0.j, 0.70710678+0.j,
       0.70710678+0.j, 0.70710678+0.j], dtype=complex128)

In [100]:
import numpy as np
import spacecore as sc

sc.set_context('numpy', dtype='float64')

X = sc.DenseVectorSpace((3,))
Y = sc.DenseVectorSpace((2,))

A = np.array(
    [[1.0, 2.0, 3.0],
     [0.0, 1.0, 0.0]]
)
linop = sc.DenseLinOp(
    A,
    dom=X,
    cod=Y,
)

x = X.ctx.asarray([1.0, 0.0, -1.0])
y = linop.apply(x)

print(y)

[-2.  0.]


In [101]:
herm =  sc.HermitianSpace(d).convert('jax')

X = herm.ctx.asarray(np.diag(np.arange(d)))
herm.check_member(X)

In [102]:
def phi(x):
    x = jnp.asarray(x)
    return jnp.where(
        x > 0,
        x * jnp.log(jnp.where(x > 0, x, 1)),
        0,
    )

In [103]:
herm.spectral_apply(X, phi)

Array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.38629436,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  3.29583687,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  5.54517744,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         8.04718956,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , 10.75055682,  0.        

In [104]:
from array_api_compat import get_namespace
import numpy as np

In [105]:
A = np.array((0., 1.))
xp = get_namespace(A)

In [106]:
dom.field

'complex'